
# Homogeneous isotropic turbulence with the LSM-1

Validation of the LSM-1 model (Random Displacement Model) on the reference
test case of a point source dispersion in a stationary homogeneous isotropic
turbulent flow, i.e. with a zero mean velocity and uniform turbulent fields.

The Monte-Carlo statistics extracted from the particle set are compared with
the exact second-order moment of the particle position and with the exact
concentration field of a point source.

**Model equations** (zero mean fluid velocity, uniform diffusivity, 2D):

\begin{align}dx_{p,i} = \sqrt{2K}\, dW_i,\end{align}

where $K$ is the turbulent diffusivity. The LSM-1 only resolves the
particle position: it is the diffusive limit ($t \gg T_L$) of the
LSM-3, and is therefore only meaningful at time scales much larger than the
Lagrangian timescale $T_L$. Only the **diffusive regime** is thus
validated here.

**Exact solution**: with the deterministic initial condition
$x_p = 0$ (point source), $x_p$ is a Gaussian random variable
with zero mean and

\begin{align}\langle x_p^2 \rangle = 2 K t,\end{align}

and the concentration generated by a point source of mass $M$ is

\begin{align}c(r, t) = \dfrac{M}{4 \pi K t} \exp \left( -\dfrac{r^2}{4 K t} \right).\end{align}

Two closures of the diffusivity are tested:

- **constant diffusivity** (``diffusion_model=1``), with
  $K = U_\alpha^2 T_L$, i.e. the diffusive limit of the LSM-3, 
  so that the LSM-1 and LSM-3 examples share the same long-time dispersion coefficient,
- **k-epsilon closure** (``diffusion_model=2``), with
  $K_t = C_\mu k^2/(\sigma_c \varepsilon)$ computed from the uniform
  turbulent fields.

Both cases use $\Delta t = 200\,T_L$ and $t_f = 24000\,T_L$, as
in the diffusive cases of the LSM-2 and LSM-3 examples.

<div class="alert alert-info"><h4>Note</h4><p>The two closures do not give the same diffusivity. With the CyLag
    definitions of $T_L$ and $U_\alpha^2$, the Langevin model
    gives $U_\alpha^2 T_L = C_0/(2(1/2+3C_0/4)^2)\, k^2/\varepsilon
    \approx 0.31\, k^2/\varepsilon$, while the k-epsilon closure gives
    $C_\mu/\sigma_c\, k^2/\varepsilon = 0.125\, k^2/\varepsilon$.</p></div>


In [ ]:
import os
import numpy as np
import matplotlib.pylab as plt
from matplotlib.tri import Triangulation

import cylag

#sphinx_gallery_thumbnail_number = 3

## Physical parameters

The turbulence is homogeneous, isotropic and stationary, with the same
parameters as in the LSM-3 example: the Lagrangian timescale is 1 s and the
turbulent velocity scale is 1 m/s, using the CyLag definitions
$T_L = (k/\varepsilon)/(1/2 + 3C_0/4)$ and
$U_\alpha^2 = C_0 \varepsilon T_L/2$.



In [ ]:
C0KOLM = 1.2   # Kolmogorov constant (cylag/core/constants.pyx)
CMU = 0.09     # k-epsilon constant (cylag/core/constants.pyx)
SIGC = 0.72    # turbulent Schmidt number (cylag/cylag_dico.py)

UALPHA = 1.0                       # turbulent velocity scale (m/s)
TL = 1.0                           # Lagrangian timescale (s)
EPS = 2.*UALPHA**2/(C0KOLM*TL)     # dissipation rate (m2/s3)
TKE = (0.5 + 0.75*C0KOLM)*TL*EPS   # turbulent kinetic energy (m2/s2)

# diffusivities of the two closures
K_LSM3 = UALPHA**2*TL              # diffusive limit of the LSM-3 (m2/s)
K_KEPS = (CMU/SIGC)*TKE**2/EPS     # k-epsilon closure (m2/s)

# Number of stochastic particles released from the point source
NPART = 20000

# Number of trajectories stored for illustration
NTRAJ = 100

print(" ~~> dissipation     = {:.4f} m2/s3".format(EPS))
print(" ~~> turbulent energ = {:.4f} m2/s2".format(TKE))
print(" ~~> T_L             = {:.4e} s".format(TL))
print(" ~~> K (U_a^2 T_L)   = {:.4f} m2/s".format(K_LSM3))
print(" ~~> K_t (k-eps)     = {:.4f} m2/s".format(K_KEPS))

## Exact solution of the LSM-1 system




In [ ]:
def lsm1_moments(t, kdiff):
    """ Exact second-order moment <xp^2> = 2*K*t of a point source """
    return 2.*kdiff*np.asarray(t, dtype='d')


def lsm1_concentration(r, t, kdiff, mass=1.):
    """ Exact concentration of a 2D point source of mass M """
    return mass/(4.*np.pi*kdiff*t)*np.exp(-r**2/(4.*kdiff*t))

## Eulerian field set

A coarse triangulation of a square domain is used: the mean velocity is zero
and the turbulent fields are uniform, so the mesh only has to be large enough
to contain the particle cloud during the whole simulation.




In [ ]:
def build_field_set(domain_size, npts=21):
    """ Uniform (zero velocity, constant k and epsilon) 2D field set """
    dx = 2.*domain_size/(npts - 1)

    # the point source at (0, 0) lies neither on a mesh node nor on an edge
    xg = np.linspace(-domain_size, domain_size, npts) + 0.137*dx
    yg = np.linspace(-domain_size, domain_size, npts) + 0.271*dx
    xx, yy = np.meshgrid(xg, yg)

    meshx = np.ascontiguousarray(xx.ravel(), dtype='d')
    meshy = np.ascontiguousarray(yy.ravel(), dtype='d')

    tri = Triangulation(meshx, meshy)
    triangles = np.ascontiguousarray(tri.triangles, dtype='int32')
    cylagtri = cylag.TriangularMesh(meshx, meshy, triangles)

    nnodes = meshx.shape[0]
    elevation = np.ones((1, nnodes), dtype='d')
    velocity_x = np.zeros((1, nnodes), dtype='d')
    velocity_y = np.zeros((1, nnodes), dtype='d')

    # k-epsilon fields, required by the k-epsilon closure
    opt_fields = np.empty((2, 1, nnodes), dtype='d')
    opt_fields[0, 0, :] = TKE
    opt_fields[1, 0, :] = EPS

    fset = cylag.EulerianFieldSet(
        triangular_mesh=cylagtri,
        dim=2, nlayers=1,
        times=np.zeros(1),
        elevation=elevation,
        velocity_x=velocity_x,
        velocity_y=velocity_y,
        opt_fields=opt_fields,
        opt_fields_names=['TURBULENT ENERG.', 'DISSIPATION'],
        trifinder=tri.get_trifinder())

    return fset

## Monte-Carlo simulation

All the particles are released at the origin. The moment is averaged over
the particles and over the two directions x and y, since the turbulence is
isotropic. The first ``NTRAJ`` trajectories are stored for illustration.




In [ ]:
def run_case(time_step, final_time, domain_size, diffusion_model,
             npart=NPART, ntraj=NTRAJ, npts=21):
    """ Run CyLag LSM-1 and return <xp^2>, trajectories and final positions """
    fset = build_field_set(domain_size, npts=npts)

    # point source: all the particles are released at the origin
    position = np.zeros((npart, 2), dtype='d')
    pset = cylag.LagrangianParticleSet(position, dim=2)

    parameters = {
        'final_time': final_time,
        'time_step': time_step,
        'frozen_eulerian_fields': True,
        'model': 1,
        'time_scheme': 1,
        'diffusion_model': diffusion_model,
        'horizontal_diffusivity': K_LSM3,
        'vertical_diffusivity': 0.,
        'schmidt_number': SIGC,
        'boundary_conditions': False,
        'listing': False,
        'output_file': False,
        }

    solver = cylag.Solver(fset, pset, parameters)

    nt = solver.simutime.nt
    times = np.zeros(nt + 1)
    xx = np.zeros(nt + 1)
    traj = np.zeros((nt + 1, ntraj, 2))

    for ite in range(nt):
        solver.forward(time_step)
        xp, _, _ = pset.get_state()
        # get_state sorts the particles, tags restore the initial ordering
        order = np.argsort(np.asarray(pset.tag)[:pset.npart_active])
        times[ite+1] = solver.simutime.time
        xx[ite+1] = np.mean(xp*xp)
        traj[ite+1] = xp[order[:ntraj]]

    if pset.npart_active != npart:
        print("WARNING: {} particles left the domain".format(
            npart - pset.npart_active))

    return times, xx, traj, np.array(xp[order])

## Post-processing utilities

The 99% confidence interval of the Monte-Carlo estimator of
$\langle x_p^2\rangle$ is obtained from the Gaussian relation
$\text{Var}(x_p^2) = 2\langle x_p^2\rangle^2$, with
$nsample = 2*npart$ independent samples (npart particles times two
directions). The same relative half-width applies to the dispersion
coefficient estimator $K_{MC} = \langle x_p^2\rangle/(2t)$.




In [ ]:
def confidence_interval(ana, npart, quantile=2.576):
    """ Half-width of the 99% confidence interval of the MC estimator """
    nsample = 2.*npart
    return quantile*np.sqrt(2./nsample)*ana


def plot_case(times, xx, ana, half, kdiff, fname, title):
    """ Plot <xp^2> and K_MC = <xp^2>/(2t) against the analytical solution """
    cylag.set_rcparams()
    colors, _, _ = cylag.get_default_color_palette()
    tstar = times/TL
    kmc = xx[1:]/(2.*times[1:])
    khalf = half[1:]/(2.*times[1:])

    fig, axes = plt.subplots(1, 2, figsize=(11., 4.2))
    fig.suptitle(title)

    ax = axes[0]
    scale = UALPHA**2*TL**2
    ax.plot(tstar, ana/scale, c='k', lw=1.5, label="Analytical")
    ax.plot(tstar, (ana + half)/scale, c='0.6', lw=0.8, ls='--',
            label="$99\\%$ confidence interval")
    ax.plot(tstar, (ana - half)/scale, c='0.6', lw=0.8, ls='--')
    ax.plot(tstar, xx/scale, c=colors[2], lw=0., marker='x', markersize=4,
            label="CyLag (LSM-1)")
    ax.set_xlabel("$t/T_L$")
    ax.set_ylabel("$\\langle x_p^2 \\rangle / (U_\\alpha T_L)^2$")
    ax.legend(loc='best', fontsize=8)

    ax = axes[1]
    ax.plot(tstar[1:], np.ones_like(kmc), c='k', lw=1.5)
    ax.plot(tstar[1:], 1. + khalf/kdiff, c='0.6', lw=0.8, ls='--')
    ax.plot(tstar[1:], 1. - khalf/kdiff, c='0.6', lw=0.8, ls='--')
    ax.plot(tstar[1:], kmc/kdiff, c=colors[2], lw=0., marker='x',
            markersize=4)
    ax.set_xlabel("$t/T_L$")
    ax.set_ylabel("$\\langle x_p^2 \\rangle / (2 K t)$")
    ax.set_ylim(0.97, 1.03)

    plt.tight_layout()
    os.makedirs("figs", exist_ok=True)
    plt.savefig(os.path.join("figs", fname), dpi=200, format="png")
    plt.show()


def print_errors(times, xx, ana, half, kdiff):
    """ Print the relative error on <xp^2> and the dispersion coefficient """
    mask = times > 0.
    err = np.abs(xx[mask] - ana[mask])/ana[mask]
    inside = np.abs(xx[mask] - ana[mask]) <= half[mask]
    print(" ~~> <xx> : mean = {:8.2e} ; max = {:8.2e} ; "
          "within 99% CI : {:5.1f}%".format(
              np.mean(err), np.max(err), 100.*np.mean(inside)))
    kmc = xx[mask]/(2.*times[mask])
    print(" ~~> dispersion coefficient K = {:.4f} m2/s "
          "(theory: {:.4f} m2/s)".format(np.mean(kmc), kdiff))

## Case 1 : constant diffusivity, K = U_alpha^2 T_L

The diffusivity is set to the diffusive limit of the LSM-3,
$K = U_\alpha^2 T_L$, so that the dispersion coefficient is the same
as in the diffusive case of the LSM-3 example.




In [ ]:
dt_dif = 200.*TL
tf_dif = 24000.*TL

times_cst, xx_cst, traj_cst, pos_cst = run_case(
    dt_dif, tf_dif, domain_size=2000., diffusion_model=1)
ana_cst = lsm1_moments(times_cst, K_LSM3)
half_cst = confidence_interval(ana_cst, NPART)

plot_case(times_cst, xx_cst, ana_cst, half_cst, K_LSM3,
          "cylag_lsm1_hit_constant.png",
          "LSM-1, constant diffusivity - $\\Delta t = 200\\,T_L$")

print_errors(times_cst, xx_cst, ana_cst, half_cst, K_LSM3)

## Case 2 : k-epsilon closure

The diffusivity is now computed from the uniform turbulent fields with the
k-epsilon closure $K_t = C_\mu k^2/(\sigma_c \varepsilon)$.




In [ ]:
times_ke, xx_ke, _, _ = run_case(
    dt_dif, tf_dif, domain_size=2000., diffusion_model=2)
ana_ke = lsm1_moments(times_ke, K_KEPS)
half_ke = confidence_interval(ana_ke, NPART)

plot_case(times_ke, xx_ke, ana_ke, half_ke, K_KEPS,
          "cylag_lsm1_hit_keps.png",
          "LSM-1, k-epsilon closure - $\\Delta t = 200\\,T_L$")

print_errors(times_ke, xx_ke, ana_ke, half_ke, K_KEPS)

## Illustration : particle trajectories

Trajectories of the first particles of case 1 and final positions of the
particle cloud. The circles are the iso-lines $r = \sigma, 2\sigma,
3\sigma$ of the exact Gaussian distribution, with
$\sigma^2 = 2 K t_f$.




In [ ]:
sigf = np.sqrt(2.*K_LSM3*times_cst[-1])
theta = np.linspace(0., 2.*np.pi, 361)

cylag.set_rcparams()
fig, ax = plt.subplots(1, 1, figsize=(8., 8.))
ax.set_aspect('equal')

ax.plot(pos_cst[:, 0], pos_cst[:, 1], c='0.5', lw=0., marker='o',
        markersize=0.5, label="$x_p(t_f)$")

for i in range(NTRAJ):
    ax.plot(traj_cst[:, i, 0], traj_cst[:, i, 1], lw=0.5)

ax.plot(0., 0., c='b', lw=0., marker='o', markersize=5., label="$x_p(0)$")
for n in range(1, 4):
    ax.plot(n*sigf*np.cos(theta), n*sigf*np.sin(theta), c='r', lw=1.,
            ls='--')

ax.set_xlabel("$x$ (m)")
ax.set_ylabel("$y$ (m)")
ax.set_xlim([-4.*sigf, 4.*sigf])
ax.set_ylim([-4.*sigf, 4.*sigf])
ax.legend(loc='upper right')
plt.savefig(os.path.join("figs", "cylag_lsm1_hit_traj.png"), dpi=200,
            format="png")
plt.show()

## Density estimate at final time

The particle concentration at final time is estimated with the Gaussian
kernel smoothing (``kernel=5``) on a regular grid, with a total mass
$M = 1$. This kernel adds a variance $h^2/8$ per direction, so
the expected estimate is the exact Gaussian with
$\sigma^2 \to \sigma^2 + h^2/8$ (up to a ~2% kernel truncation).




In [ ]:
h = 0.5*sigf
massp = 1./pos_cst.shape[0]

xg = np.linspace(-4.*sigf, 4.*sigf, 61)
xx_g, yy_g = np.meshgrid(xg, xg)
rr_g = np.sqrt(xx_g**2 + yy_g**2)

# the ghost mesh estimator sorts the positions in place: pass a copy
density = cylag.compute_density_2D_KS_GM(
    xx_g.flatten(), yy_g.flatten(), np.ascontiguousarray(pos_cst.copy()),
    massp, h, 5)
density = np.asarray(density).reshape(np.shape(xx_g))

tf = times_cst[-1]
exact = lsm1_concentration(rr_g, tf, K_LSM3)

rr = np.linspace(0., 4.*sigf, 201)
exact_r = lsm1_concentration(rr, tf, K_LSM3)
smooth_r = lsm1_concentration(rr, tf + h**2/(16.*K_LSM3), K_LSM3)

cylag.set_rcparams()
colors, _, _ = cylag.get_default_color_palette()
fig, axes = plt.subplots(1, 2, figsize=(13., 5.5))

ax = axes[0]
ax.set_aspect('equal')
levels = np.linspace(0., np.max(exact), 11)
img = ax.contourf(xg, xg, density, cmap="coolwarm", levels=levels,
                  extend="max")
ax.contour(xg, xg, density, linewidths=0.5, colors='k', levels=levels[1:])
ax.contour(xg, xg, exact, linewidths=1., colors='w', linestyles='--',
           levels=levels[1:])
fig.colorbar(img, ax=ax, label="$c$ (m$^{-2}$)")
ax.set_xlabel("$x$ (m)")
ax.set_ylabel("$y$ (m)")
ax.set_title("CyLag (filled) and exact (dashed)")

ax = axes[1]
ax.plot(rr_g.flatten()/sigf, density.flatten(), c=colors[2], lw=0.,
        marker='x', markersize=3, label="CyLag (LSM-1, KS)")
ax.plot(rr/sigf, exact_r, c='k', lw=1.5, label="Analytical")
ax.plot(rr/sigf, smooth_r, c='0.4', lw=1., ls='--',
        label="Analytical smoothed by the kernel")
ax.set_xlabel("$r/\\sigma$")
ax.set_ylabel("$c$ (m$^{-2}$)")
ax.set_xlim([0., 4.])
ax.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join("figs", "cylag_lsm1_hit_density.png"), dpi=200,
            format="png")
plt.show()

ic = np.argmin(rr_g)
print(" ~~> peak concentration: CyLag = {:.4e} ; smoothed exact = {:.4e} ; "
      "exact = {:.4e} m-2".format(density.flatten()[ic], smooth_r[0],
                                  exact_r[0]))

Clean:



In [ ]:
del times_cst, xx_cst, traj_cst, pos_cst, ana_cst, half_cst
del times_ke, xx_ke, ana_ke, half_ke